# Experiment

## Import libraries

In [7]:
import pandas as pd

file_path = "exchange_rate_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="date_str",
    value_name="value",
)

# Filter only valid dd/mm/yyyy
df = df[df["date_str"].str.match(r"\d{2}/\d{2}/\d{4}")]

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Convert to datetime
df["date"] = pd.to_datetime(
    df["date_str"], format="%d/%m/%Y", errors="coerce"
)

# Extract year, month, day
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# Pivot table to wide format
df = df.pivot_table(
    index=["year", "month", "day"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by date
df = df.sort_values(["year", "month", "day"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

# Rename columns
df.rename(columns={"central_rate_from_04012016": "central_rate"}, inplace=True)

df

Chỉ tiêu,year,month,day,central_rate
0,2016,1,4,21896.0
1,2016,1,5,21907.0
2,2016,1,6,21907.0
3,2016,1,7,21919.0
4,2016,1,8,21909.0
...,...,...,...,...
2830,2025,8,6,25232.0
2831,2025,8,7,25239.0
2832,2025,8,8,25228.0
2833,2025,8,9,25228.0


In [8]:
df.columns

Index(['year', 'month', 'day', 'central_rate'], dtype='object', name='Chỉ tiêu')